# Estatísticas via Understat + merge com o Transfermarkt

O Understat expõe os dados por um endpoint AJAX (`POST /main/getPlayersStats/`):
uma requisição por temporada, sem scraping de HTML.

O merge é a parte delicada — os nomes vêm em formatos diferentes dos dois lados,
então é feito um fuzzy join por temporada. Quatro regras evitam os erros da
versão anterior:

- **Atribuição 1-para-1** — cada jogador do Understat é usado no máximo uma vez
  por temporada. Antes, `extractOne` era chamado linha a linha sem remover quem
  já tinha sido usado, e três jogadores diferentes podiam herdar a mesma linha
  de estatísticas.
- **Desempate por clube** — quando os nomes empatam (três "Diego López" em
  2021), o clube decide qual é o certo.
- **Clube como confirmação** — nome inexato (85–99) só passa se o clube também
  bater. Foi o que separou 'Saúl García' (Deportivo) de 'Raúl García' (Leganés).
- **Um limiar só** — `MATCH_THRESHOLD` é aplicado dentro do join. Antes havia
  80 na função e 85 numa célula posterior, e a anulação dos 80–84 acabou não
  entrando no arquivo salvo.

Jogadores transferidos no meio da temporada aparecem em mais de uma linha no
painel do Transfermarkt; aqui fica só a última.

In [25]:
import os
import re
import time
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from rapidfuzz import fuzz, process


def find_project_root(marker: str = "CLAUDE.md") -> Path:
    """Sobe a partir do diretório atual até achar a raiz do repositório."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(f"não encontrei {marker} em {here} nem acima")


BASE_PATH = Path(os.environ.get("TRANSFER_EDGE_ROOT") or find_project_root())
DATA_DIR = BASE_PATH / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

TM_PATH = DATA_DIR / "players_panel.parquet"
UNDERSTAT_PATH = DATA_DIR / "understat_stats.parquet"
PANEL_PATH = DATA_DIR / "panel_final.parquet"

YEARS = list(range(2017, 2026))   # alinhado com o Transfermarkt (notebook 01)
REQUEST_DELAY = 1.5               # respeita o rate limit

# Limiar único de match, aplicado dentro do join — não existe segunda etapa de
# anulação. Score do nome abaixo disso: as stats ficam NaN.
MATCH_THRESHOLD = 85

# Um nome inexato (85-99) precisa que o clube também bata. Nome exato (100)
# passa sem isso, porque quem foi transferido no meio do ano legitimamente tem
# clube diferente nos dois lados. Calibrado olhando os matches reais: abaixo de
# 60 eram todos jogadores trocados, acima de 78 todos corretos.
TEAM_FLOOR = 60

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ),
    "Referer": "https://understat.com/",
    "X-Requested-With": "XMLHttpRequest",
    "Content-Type": "application/x-www-form-urlencoded",
}

print(f"raiz do projeto: {BASE_PATH}")

raiz do projeto: /Users/luccagazotto/Documents/Personal/ML Projects/Soccer player/Soccer_player_market_value


## 1. Coleta

O Understat usa o ano de início da temporada (2017 = 2017-18), igual à convenção
do Transfermarkt (`saison_id=2017`). Pedimos todos os anos que o notebook 01
cobre; os que ainda não existem no Understat são registrados, e as linhas
correspondentes do TM ficam sem stats em vez de sumirem em silêncio.

In [26]:
def get_understat_season(year: int) -> pd.DataFrame:
    """Uma temporada inteira da La Liga em uma requisição."""
    resp = requests.post(
        "https://understat.com/main/getPlayersStats/",
        headers=HEADERS,
        data={"league": "La_liga", "season": str(year)},
        timeout=30,
    )
    resp.raise_for_status()
    players = resp.json().get("players") or []
    df = pd.DataFrame(players)
    if df.empty:
        return df
    df["season_year"] = year
    return df


seasons, sem_dados = [], []
for y in YEARS:
    df_s = get_understat_season(y)
    if df_s.empty:
        sem_dados.append(y)
        print(f"{y}-{y + 1}: sem dados")
    else:
        seasons.append(df_s)
        print(f"{y}-{y + 1}: {len(df_s)} jogadores")
    time.sleep(REQUEST_DELAY)

understat_raw = pd.concat(seasons, ignore_index=True)
print(f"\nTotal bruto: {understat_raw.shape}")

if sem_dados:
    print(f"Temporadas sem cobertura no Understat: {sem_dados}")
    print("  -> as linhas do Transfermarkt desses anos ficam com stats NaN (esperado)")

2017-2018: 557 jogadores
2018-2019: 531 jogadores
2019-2020: 556 jogadores
2020-2021: 570 jogadores
2021-2022: 603 jogadores
2022-2023: 584 jogadores
2023-2024: 598 jogadores
2024-2025: 590 jogadores
2025-2026: 600 jogadores

Total bruto: (5189, 19)


## 2. Limpeza e normalização

`name_norm` e `team_norm` são criados **antes** de salvar, para que o arquivo em
disco tenha as colunas que o merge usa. Na versão anterior o parquet era salvo
antes da normalização, e o schema documentado não batia com o arquivo.

Nomes e clubes usam normalizações diferentes de propósito. Em nome, pontuação é
apagada. Em clube, pontuação vira espaço: o Understat escreve o time de quem foi
transferido como `'Getafe,Villarreal'`, e apagar a vírgula grudaria tudo num
token só (`'getafevillarreal'`), zerando a comparação.

In [27]:
def _strip_accents(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    return "".join(c for c in text if not unicodedata.combining(c))


def normalize_name(name) -> str:
    """Remove acentos, pontuação e caixa — maximiza o matching de nomes."""
    if not isinstance(name, str):
        return ""
    name = re.sub(r"[^a-z ]", "", _strip_accents(name).lower().strip())
    return re.sub(r"\s+", " ", name)


def normalize_team(team) -> str:
    """Igual, mas pontuação vira espaço: 'Getafe,Villarreal' -> dois tokens."""
    if not isinstance(team, str):
        return ""
    team = re.sub(r"[^a-z]+", " ", _strip_accents(team).lower())
    return re.sub(r"\s+", " ", team).strip()


understat_df = understat_raw.rename(columns={
    "id": "understat_id",
    "season_year": "year",
    "player_name": "player_name_us",
    "team_title": "team_us",
    "position": "position_us",
    "time": "minutes",
}).copy()

numeric_cols = ["games", "minutes", "goals", "assists", "shots", "key_passes",
                "yellow_cards", "red_cards", "npg", "xG", "xA", "npxG",
                "xGChain", "xGBuildup"]
for col in [c for c in numeric_cols if c in understat_df.columns]:
    understat_df[col] = pd.to_numeric(understat_df[col], errors="coerce")

understat_df["name_norm"] = understat_df["player_name_us"].apply(normalize_name)
understat_df["team_norm"] = understat_df["team_us"].apply(normalize_team)

understat_df.to_parquet(UNDERSTAT_PATH, index=False)
print(f"Salvo {UNDERSTAT_PATH.name}: {understat_df.shape}")
print(f"Colunas: {list(understat_df.columns)}")

Salvo understat_stats.parquet: (5189, 21)
Colunas: ['understat_id', 'player_name_us', 'games', 'minutes', 'goals', 'xG', 'assists', 'xA', 'shots', 'key_passes', 'yellow_cards', 'red_cards', 'position_us', 'team_us', 'npg', 'npxG', 'xGChain', 'xGBuildup', 'year', 'name_norm', 'team_norm']


## 3. Transferências no meio da temporada

O painel do Transfermarkt tem uma linha por (temporada, clube, jogador), então
quem trocou de clube durante o ano aparece duas vezes. O Understat tem uma linha
por jogador-temporada, com os números do ano inteiro — copiar isso para as duas
passagens duplicaria minutos e gols.

Mantemos **a última linha** de cada jogador-temporada.

> Sem data de transferência no painel, "última" é a última na ordem em que o
> notebook 01 raspou. Para uma ordem cronológica de verdade seria preciso
> guardar o `clubId` do snapshot de dezembro da API de valores de mercado.

In [28]:
tm_df = pd.read_parquet(TM_PATH)
tm_df["year"] = tm_df["year"].astype(int)
tm_df["player_id"] = tm_df["player_id"].astype(str)

antes = len(tm_df)
multi = tm_df.duplicated(subset=["year", "player_id"], keep=False).sum()
tm_df = tm_df.drop_duplicates(subset=["year", "player_id"], keep="last").reset_index(drop=True)

print(f"Transfermarkt: {antes} linhas")
print(f"  jogador-temporada com mais de um clube: {multi} linhas")
print(f"  mantida a última de cada -> {len(tm_df)} linhas ({antes - len(tm_df)} removidas)")

tm_df["name_norm"] = tm_df["player_name"].apply(normalize_name)
tm_df["team_norm"] = tm_df["club"].apply(normalize_team)

Transfermarkt: 6677 linhas
  jogador-temporada com mais de um clube: 327 linhas
  mantida a última de cada -> 6513 linhas (164 removidas)


## 4. Fuzzy join por temporada

`process.cdist` monta a matriz de scores de uma vez, no lugar do laço com
`extractOne` linha a linha.

- **Nome**: `token_sort_ratio`, que tolera a ordem dos tokens
  ('marc andre ter stegen' vs 'ter stegen marc andre').
- **Clube**: `token_set_ratio`, que lida com subconjuntos
  ('Espanyol' dentro de 'RCD Espanyol Barcelona').

Um par é candidato se o nome atinge `MATCH_THRESHOLD` **e** — quando o nome não é
exato — o clube atinge `TEAM_FLOOR`. Os candidatos são ordenados por score do
nome, com o clube desempatando, e atribuídos gulosamente: cada lado entra em no
máximo um par. O nome continua mandando; o clube só separa empates e barra os
quase-homônimos de times diferentes.

Os ~24% sem match são jogadores emprestados fora da La Liga ou reservas que não
entraram em campo: o Understat só lista quem jogou, então NaN é o certo.

In [29]:
STAT_COLS = ["understat_id", "player_name_us", "team_us", "position_us",
             "games", "minutes", "goals", "xG", "assists", "xA", "shots",
             "key_passes", "yellow_cards", "red_cards", "npg", "npxG",
             "xGChain", "xGBuildup"]


def fuzzy_join_season(tm_s: pd.DataFrame, us_s: pd.DataFrame,
                      threshold: int = MATCH_THRESHOLD,
                      team_floor: int = TEAM_FLOOR) -> pd.DataFrame:
    """Atribuição 1-para-1 entre Transfermarkt e Understat numa temporada."""
    tm_s = tm_s.reset_index(drop=True)
    us_s = us_s.reset_index(drop=True)
    vazio = {c: np.nan for c in STAT_COLS}

    if us_s.empty:
        return tm_s.assign(match_score=np.nan, **vazio)

    name_scores = process.cdist(
        tm_s["name_norm"], us_s["name_norm"],
        scorer=fuzz.token_sort_ratio, dtype=np.float32, workers=-1,
    )
    team_scores = process.cdist(
        tm_s["team_norm"], us_s["team_norm"],
        scorer=fuzz.token_set_ratio, dtype=np.float32, workers=-1,
    )

    # Nome exato dispensa o clube; nome inexato precisa de confirmação.
    candidato = (name_scores >= threshold) & (
        (name_scores >= 100) | (team_scores >= team_floor)
    )
    ti, ui = np.where(candidato)
    if len(ti) == 0:
        return tm_s.assign(match_score=np.nan, **vazio)

    # lexsort usa a ÚLTIMA chave como primária: nome manda, clube desempata.
    order = np.lexsort((-team_scores[ti, ui], -name_scores[ti, ui]))

    usados_tm, usados_us, pares = set(), set(), []
    for k in order:
        t, u = int(ti[k]), int(ui[k])
        if t in usados_tm or u in usados_us:
            continue
        usados_tm.add(t)
        usados_us.add(u)
        pares.append((t, u, float(name_scores[t, u])))

    link = pd.DataFrame(pares, columns=["_tm_row", "_us_row", "match_score"])
    stat_cols = [c for c in STAT_COLS if c in us_s.columns]
    out = tm_s.reset_index(names="_tm_row")
    us_slim = us_s.reset_index(names="_us_row")[["_us_row"] + stat_cols]

    out = out.merge(link, on="_tm_row", how="left").merge(us_slim, on="_us_row", how="left")
    return out.drop(columns=["_tm_row", "_us_row"])

In [30]:
todas = []
for year in sorted(tm_df["year"].unique()):
    tm_s = tm_df[tm_df["year"] == year]
    us_s = understat_df[understat_df["year"] == year]

    merged = fuzzy_join_season(tm_s, us_s)
    matched = merged["match_score"].notna().sum()
    pct = 100 * matched / len(merged) if len(merged) else 0
    print(f"{year}: {matched}/{len(merged)} com stats ({pct:.0f}%) "
          f"— {len(us_s)} jogadores no Understat")
    todas.append(merged)

panel_final = pd.concat(todas, ignore_index=True)
print(f"\nPainel final: {panel_final.shape}")

2017: 472/654 com stats (72%) — 557 jogadores no Understat
2018: 460/657 com stats (70%) — 531 jogadores no Understat
2019: 481/686 com stats (70%) — 556 jogadores no Understat
2020: 496/716 com stats (69%) — 570 jogadores no Understat
2021: 535/764 com stats (70%) — 603 jogadores no Understat
2022: 526/739 com stats (71%) — 584 jogadores no Understat
2023: 543/757 com stats (72%) — 598 jogadores no Understat
2024: 531/767 com stats (69%) — 590 jogadores no Understat
2025: 550/773 com stats (71%) — 600 jogadores no Understat

Painel final: (6513, 37)


## 5. Verificação

As três invariantes que a versão anterior violava. A checagem de reuso é feita
pelo `understat_id`, não pelo nome: o Understat tem 30 casos de dois jogadores
**diferentes** com o mesmo nome na mesma temporada (dois 'Raúl García' em 2017,
por exemplo), e conferir por nome acusaria isso como erro.

In [31]:
com_stats = panel_final[panel_final["match_score"].notna()]

reusados = com_stats.groupby(["year", "understat_id"]).size().pipe(lambda s: s[s > 1])
abaixo = (com_stats["match_score"] < MATCH_THRESHOLD).sum()
dup_jogador = panel_final.duplicated(subset=["year", "player_id"]).sum()

print(f"Linhas com stats             : {len(com_stats)}")
print(f"Score mínimo                 : {com_stats['match_score'].min():.1f}")
print(f"Abaixo do limiar ({MATCH_THRESHOLD})       : {abaixo}")
print(f"Jogador do Understat reusado : {len(reusados)}")
print(f"Jogador-temporada duplicado  : {dup_jogador}")

assert abaixo == 0, "limiar não aplicado"
assert len(reusados) == 0, f"atribuição não é 1-para-1: {reusados.head()}"
assert dup_jogador == 0, "jogador-temporada duplicado"
print("\nOK — 1-para-1, limiar aplicado, uma linha por jogador-temporada.")

Linhas com stats             : 4594
Score mínimo                 : 85.7
Abaixo do limiar (85)       : 0
Jogador do Understat reusado : 0
Jogador-temporada duplicado  : 0

OK — 1-para-1, limiar aplicado, uma linha por jogador-temporada.


In [32]:
panel_final.to_parquet(PANEL_PATH, index=False)
print(f"Salvo {PANEL_PATH.name}: {panel_final.shape}")
print(f"Colunas: {list(panel_final.columns)}")
panel_final.head()

Salvo panel_final.parquet: (6513, 37)
Colunas: ['year', 'club', 'player_id', 'player_name', 'link_html', 'birth_date', 'image_url', 'nationality', 'height', 'foot', 'position', 'market_value', 'value_date', 'age', 'market_value_imputed', 'age_imputed', 'name_norm', 'team_norm', 'match_score', 'understat_id', 'player_name_us', 'team_us', 'position_us', 'games', 'minutes', 'goals', 'xG', 'assists', 'xA', 'shots', 'key_passes', 'yellow_cards', 'red_cards', 'npg', 'npxG', 'xGChain', 'xGBuildup']


,year,club,player_id,player_name,link_html,birth_date,image_url,nationality,height,foot,...,assists,xA,shots,key_passes,yellow_cards,red_cards,npg,npxG,xGChain,xGBuildup
0,2017,Getafe CF,102423,Leandro Cabrera,https://www.transfermarkt.us/leandro-cabrera/p...,1991-06-17,https://img.a.transfermarkt.technology/portrai...,Uruguay,1.91,left,...,0.0,0.121607,2.0,2.0,3.0,0.0,0.0,0.688738,1.133077,1.011469
1,2017,Sevilla FC,102744,Sébastien Corchia,https://www.transfermarkt.us/sebastien-corchia...,1990-11-01,https://img.a.transfermarkt.technology/portrai...,France,1.75,right,...,1.0,0.534557,5.0,5.0,4.0,0.0,0.0,0.324520,1.887542,1.212583
2,2017,Sevilla FC,105899,Guido Pizarro,https://www.transfermarkt.us/guido-pizarro/pro...,1990-02-26,https://img.a.transfermarkt.technology/portrai...,Argentina,1.85,right,...,0.0,0.933196,13.0,8.0,9.0,0.0,1.0,0.915141,6.310068,5.370854
3,2017,Deportivo Alavés,106825,Enzo Zidane,https://www.transfermarkt.us/enzo-zidane/profi...,1995-03-24,https://img.a.transfermarkt.technology/portrai...,France,1.85,both,...,0.0,0.023231,0.0,1.0,0.0,0.0,0.0,0.000000,0.061625,0.061625
4,2017,Atlético de Madrid,107010,Stefan Savić,https://www.transfermarkt.us/stefan-savic/prof...,1991-01-08,https://img.a.transfermarkt.technology/portrai...,Montenegro,1.88,right,...,0.0,0.165848,3.0,3.0,6.0,0.0,0.0,0.205908,3.473261,3.330281
